In [1]:
import pandas as pd
import numpy as np

In [2]:
import sys; sys.path.append("..")
from src.simulator import assign_preferences, _rates_matrix, choice_weights, choose_lender, simulate_market, winners_curse

## Preferences

In [3]:
applicants = pd.DataFrame({"id": [1, 2, 3, 4, 5, 6]})
with_pref = assign_preferences(applicants, seed=42)
with_pref

,id,pref
0,1,1
1,2,3
2,3,3
3,4,2
4,5,2
5,6,1


## Validate rates

In [4]:
offers = pd.DataFrame({"r1": [0.05, np.nan], "r2": [0.06, 0.12], "r3": [0.08, 0.09]})
_rates_matrix(offers)

array([[0.05, 0.06, 0.08],
       [ nan, 0.12, 0.09]])

## Choice of weight for each rate

In [5]:
rates = np.array([[0.05, 0.06, 0.08]] * 3)   # same offers for 3 applicants
pref  = np.array([1, 2, 3])                   # one applicant of each type
choice_weights(rates, pref)

array([[1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [6]:
choice_weights(np.array([[0.07, 0.05, 0.05]]), [1]) # tie in preference

array([[0.33333333, 0.33333333, 0.33333333]])

In [7]:
choice_weights(np.array([[np.nan, 0.10, 0.30]]), [1]) # choose lowest rate

array([[0., 1., 0.]])

In [8]:
choice_weights(np.array([[np.nan, np.nan, np.nan]]), [1]) # no offer

array([[0., 0., 0.]])

## Choice of lender

In [9]:
rng = np.random.default_rng(0)
choose_lender(np.array([[0.05, 0.06, 0.08]] * 3), [1, 2, 3], rng=rng)

array([1, 2, 1])

## Simulate market

In [4]:
offers = pd.DataFrame(
    {"r1": [0.05, 0.10, np.nan, 0.20, np.nan],
     "r2": [0.06, 0.07, 0.12,   np.nan, np.nan],
     "r3": [0.08, 0.07, 0.09,   np.nan, np.nan]},
    index=[1, 2, 3, 4, 5])
defaults = [0, 1, 0, 1, 0]

loans, summary = simulate_market(offers, defaults, mode="expected")

In [5]:
loans

,r1,r2,r3,default,w1,w2,w3,profit1,profit2,profit3,exp_rate_taken,loan_made_prob
1,0.05,0.06,0.08,0,0.666667,0.333333,0.0,333.333333,200.0,0.0,0.053333,1.0
2,0.10,0.07,0.07,1,0.000000,0.500000,0.5,-0.000000,-5000.0,-5000.0,0.070000,1.0
3,NaN,0.12,0.09,0,0.000000,0.000000,1.0,0.000000,0.0,900.0,0.090000,1.0
4,0.20,NaN,NaN,1,1.000000,0.000000,0.0,-10000.000000,-0.0,-0.0,0.200000,1.0
5,NaN,NaN,NaN,0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0


In [6]:
summary

,n_loans,share_of_applicants,share_of_loans,avg_rate,default_rate_won,total_profit,profit_per_loan
lender,,,,,,,
1,1.666667,0.333333,0.416667,0.140000,0.600000,-9666.666667,-5800.000000
2,0.833333,0.166667,0.208333,0.066000,0.600000,-4800.000000,-5760.000000
3,1.500000,0.300000,0.375000,0.083333,0.333333,-4100.000000,-2733.333333


In [7]:
loans2, summary2 = simulate_market(offers, defaults, mode="random")

In [8]:
loans2

,r1,r2,r3,default,w1,w2,w3,profit1,profit2,profit3,exp_rate_taken,loan_made_prob,pref,chosen_lender
1,0.05,0.06,0.08,0,0.0,1.0,0.0,0.0,600.0,0.0,0.06,1.0,2,2
2,0.10,0.07,0.07,1,0.0,0.0,1.0,-0.0,-0.0,-10000.0,0.07,1.0,3,3
3,NaN,0.12,0.09,0,0.0,0.0,1.0,0.0,0.0,900.0,0.09,1.0,1,3
4,0.20,NaN,NaN,1,1.0,0.0,0.0,-10000.0,-0.0,-0.0,0.20,1.0,2,1
5,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,1,0


In [9]:
summary2

,n_loans,share_of_applicants,share_of_loans,avg_rate,default_rate_won,total_profit,profit_per_loan
lender,,,,,,,
1,1.0,0.2,0.25,0.20,1.0,-10000.0,-10000.0
2,1.0,0.2,0.25,0.06,0.0,600.0,600.0
3,2.0,0.4,0.50,0.08,0.5,-9100.0,-4550.0


# Winner's curse

In [ ]:
p3 = np.array([0.10, 0.30, 0.05, 0.40, 0.20])   # our predicted p for applicants 1 to 5
winners_curse(loans, p3, lender=3, groups=2)

,group_size,n_loans_won,win_rate,avg_pred_p,obs_default_rate,gap,se_obs,p_low,p_high
G1,3,1.0,0.333333,0.050000,0.000000,-0.05,0.000000,0.05,0.2
G2,2,0.5,0.250000,0.300000,1.000000,0.70,0.000000,0.30,0.4
overall,5,1.5,0.300000,0.133333,0.333333,0.20,0.351364,0.05,0.4
